In [2]:
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models

from tensorflow.keras.optimizers import Adam

import matplotlib.pyplot as plt

In [7]:
IMG_HEIGHT = 224
IMG_WIDTH = 224

BATCH_SIZE = 32

SEED = 42

DATASET_PATH = "/workspaces/plant_disease_detection/data/raw"

In [8]:
train_dataset = tf.keras.utils.image_dataset_from_directory(

    DATASET_PATH,

    validation_split=0.2,

    subset="training",

    seed=SEED,

    image_size=(IMG_HEIGHT, IMG_WIDTH),

    batch_size=BATCH_SIZE
)

Found 20638 files belonging to 15 classes.
Using 16511 files for training.


In [9]:
validation_dataset = tf.keras.utils.image_dataset_from_directory(

    DATASET_PATH,

    validation_split=0.2,

    subset="validation",

    seed=SEED,

    image_size=(IMG_HEIGHT, IMG_WIDTH),

    batch_size=BATCH_SIZE
)

Found 20638 files belonging to 15 classes.
Using 4127 files for validation.


In [10]:
class_names = train_dataset.class_names

NUM_CLASSES = len(class_names)

print("Number of Classes:", NUM_CLASSES)

print(class_names)

Number of Classes: 15
['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [11]:
train_dataset = train_dataset.take(100)

validation_dataset = validation_dataset.take(20)

In [13]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

validation_dataset = validation_dataset.prefetch(
    buffer_size=AUTOTUNE
)

In [14]:
learning_rates = [

    0.01,

    0.001,

    0.0001

]

In [15]:
def build_lr_model(learning_rate):

    model = models.Sequential([

        layers.Input(
            shape=(IMG_HEIGHT,
                   IMG_WIDTH,
                   3)
        ),

        layers.Rescaling(1./255),

        layers.Conv2D(
            32,
            (3,3),
            activation='relu'
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            64,
            (3,3),
            activation='relu'
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            128,
            (3,3),
            activation='relu'
        ),

        layers.MaxPooling2D((2,2)),

        layers.Flatten(),

        layers.Dense(
            128,
            activation='relu'
        ),

        layers.Dropout(0.3),

        layers.Dense(
            NUM_CLASSES,
            activation='softmax'
        )

    ])

    model.compile(

        optimizer=Adam(
            learning_rate=learning_rate
        ),

        loss='sparse_categorical_crossentropy',

        metrics=['accuracy']

    )

    return model

In [16]:
lr_results = {}

In [17]:
for lr in learning_rates:

    print(f"\nTraining with Learning Rate = {lr}")

    model = build_lr_model(lr)

    history = model.fit(

        train_dataset,

        validation_data=validation_dataset,

        epochs=3,

        verbose=1

    )

    best_accuracy = max(
        history.history['val_accuracy']
    )

    lr_results[lr] = best_accuracy


Training with Learning Rate = 0.01
Epoch 1/3


W0000 00:00:1782083736.643265    1711 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1782083736.676772    1711 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1782083736.689840    1711 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1782083737.600122    1711 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1782083740.697558    2267 cpu_allocator_impl.cc:82] Allocation of 201867264 exceeds 10% of free system memory.


100/100 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.1522 - loss: 7.9984 - val_accuracy: 0.1578 - val_loss: 2.5921
Epoch 2/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 94s 939ms/step - accuracy: 0.1666 - loss: 2.5712 - val_accuracy: 0.1484 - val_loss: 2.6042
Epoch 3/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 98s 982ms/step - accuracy: 0.1700 - loss: 2.5675 - val_accuracy: 0.1562 - val_loss: 2.5876

Training with Learning Rate = 0.001
Epoch 1/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 99s 977ms/step - accuracy: 0.2853 - loss: 2.2339 - val_accuracy: 0.3844 - val_loss: 1.9608
Epoch 2/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.5581 - loss: 1.3759 - val_accuracy: 0.6828 - val_loss: 1.0528
Epoch 3/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.6544 - loss: 1.0964 - val_accuracy: 0.7094 - val_loss: 0.8928

Training with Learning Rate = 0.0001
Epoch 1/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 100s 984ms/step - accuracy: 0.2350 - loss: 2.3962 - val_accuracy: 0.3609 - val_loss: 2.0900
Epoch 2/3
100/100 ━━━━━━━━━━━━

In [18]:
print("\nLearning Rate Tuning Results\n")

for lr, accuracy in lr_results.items():

    print(f"Learning Rate: {lr}")

    print(f"Best Validation Accuracy: {accuracy:.4f}\n")


Learning Rate Tuning Results

Learning Rate: 0.01
Best Validation Accuracy: 0.1578

Learning Rate: 0.001
Best Validation Accuracy: 0.7094

Learning Rate: 0.0001
Best Validation Accuracy: 0.5938



In [19]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.2),

    layers.RandomZoom(0.2)

])

In [20]:
dropout_rates = [

    0.2,

    0.3,

    0.5

]

In [21]:
def build_dropout_model(dropout_rate):

    model = models.Sequential([

        layers.Input(
            shape=(IMG_HEIGHT,
                   IMG_WIDTH,
                   3)
        ),

        data_augmentation,

        layers.Rescaling(1./255),

        layers.Conv2D(
            32,
            (3,3),
            activation='relu'
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            64,
            (3,3),
            activation='relu'
        ),

        layers.MaxPooling2D((2,2)),

        layers.Conv2D(
            128,
            (3,3),
            activation='relu'
        ),

        layers.MaxPooling2D((2,2)),

        layers.Flatten(),

        layers.Dense(
            128,
            activation='relu'
        ),

        layers.Dropout(dropout_rate),

        layers.Dense(
            NUM_CLASSES,
            activation='softmax'
        )

    ])

    return model

In [22]:
dropout_results = {}

In [23]:
for rate in dropout_rates:

    print(f"\nTraining with Dropout = {rate}")

    model = build_dropout_model(rate)

    model.compile(

        optimizer=Adam(
            learning_rate=0.001
        ),

        loss='sparse_categorical_crossentropy',

        metrics=['accuracy']
    )

    history = model.fit(

        train_dataset,

        validation_data=validation_dataset,

        epochs=3

    )

    best_accuracy = max(
        history.history['val_accuracy']
    )

    dropout_results[rate] = best_accuracy


Training with Dropout = 0.2
Epoch 1/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.2512 - loss: 2.3989 - val_accuracy: 0.3125 - val_loss: 2.1640
Epoch 2/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.3887 - loss: 1.9106 - val_accuracy: 0.4922 - val_loss: 1.6166
Epoch 3/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.4712 - loss: 1.6619 - val_accuracy: 0.5797 - val_loss: 1.4626

Training with Dropout = 0.3
Epoch 1/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - accuracy: 0.2269 - loss: 2.4329 - val_accuracy: 0.3187 - val_loss: 2.1566
Epoch 2/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.4050 - loss: 1.8974 - val_accuracy: 0.4266 - val_loss: 1.6992
Epoch 3/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 145s 1s/step - accuracy: 0.5109 - loss: 1.5747 - val_accuracy: 0.5375 - val_loss: 1.4300

Training with Dropout = 0.5
Epoch 1/3
100/100 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.2334 - loss: 2.4043 - val_accuracy: 0.3938 - val_loss: 1.9921
Epoch 2/3
100/100

In [26]:
print("\nDropout Tuning Results\n")

for rate, accuracy in dropout_results.items():

    print(f"Dropout Rate: {rate}")

    print(f"Best Validation Accuracy: {accuracy:.4f}\n")


Dropout Tuning Results

Dropout Rate: 0.2
Best Validation Accuracy: 0.5797

Dropout Rate: 0.3
Best Validation Accuracy: 0.5375

Dropout Rate: 0.5
Best Validation Accuracy: 0.5719



In [27]:
best_lr = max(
    lr_results,
    key=lr_results.get
)

best_dropout = max(
    dropout_results,
    key=dropout_results.get
)

print("Best Learning Rate:", best_lr)

print("Best Dropout Rate:", best_dropout)

Best Learning Rate: 0.001
Best Dropout Rate: 0.2
